In [1]:
# ============================================================================
# notebook: notebooks/06_mechanism.ipynb  (R-14: mechanism of stable-yet-fragile)
# Project: "Incidental vs. Engineered Approval"
# Goal: explain why disadvantaged approvals are MORE stable (Stability↑) yet MORE
#   fragile (NonFragility↓) — apparently contradictory. Hypothesis:
#     dense clustering  ->  consistent SHAP among similar neighbors (Stability↑)
#                      \->  sits in the 'typicality-paradox' risky region, near a
#                           non-flat boundle (Fragility↑ i.e. NonFragility↓)
#   So local density is a COMMON CAUSE of both. Tests:
#     M1. Is disadvantaged group denser (lower LowDensity) in the audit space?
#     M2. Does density DRIVE Stability (dense -> stable SHAP)?  path 1
#     M3. Does density DRIVE Fragility (dense -> fragile)?      path 2
#     M4. Mediation: does controlling density collapse the Stability & Fragility
#         group-gaps? (if yes, density mediates -> the paradox is one mechanism)
# Reads results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, load Stage-3 scored borderline set (has axes + group)
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

ROOT    = Path("..").resolve()
RESULTS = ROOT / "results"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

B = pd.read_parquet(RESULTS / "stage3_borderline_scored.parquet")
# raw density percentile (not the low-flipped one) for mechanism clarity:
B["density_pct"] = 1.0 - B["A_LowDensity"]   # recover raw density percentile
sub = B[B["GROUP"].isin(["dis_primary","advantaged"])].copy()
sub["is_dis"] = (sub["GROUP"]=="dis_primary").astype(int)
print(f"Borderline: dis_primary={int(sub.is_dis.sum())}, advantaged={int((1-sub.is_dis).sum())}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — M1: is the disadvantaged group DENSER? (the common-cause candidate)
# ─────────────────────────────────────────────────────────────────────────
dis = sub[sub.is_dis==1]; adv = sub[sub.is_dis==0]
_, p_d = stats.mannwhitneyu(dis["density_pct"], adv["density_pct"], alternative="greater")
print("M1 — density by group (raw density percentile; higher=denser):")
print(f"  dis_primary mean density = {dis['density_pct'].mean():.3f}")
print(f"  advantaged  mean density = {adv['density_pct'].mean():.3f}")
print(f"  Mann-Whitney (dis>adv) p = {p_d:.3e}  "
      f"-> {'disadvantaged IS denser' if p_d<0.05 else 'no density difference'}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — M2 & M3: does density DRIVE both Stability and Fragility?
# Regress each axis on density (within the combined borderline sub-set).
# Expect: density -> Stability positive; density -> NonFragility negative.
# ─────────────────────────────────────────────────────────────────────────
def ols(y, xcols, frame):
    X = sm.add_constant(frame[xcols])
    return sm.OLS(frame[y].values, X).fit()

print("M2 — density -> Stability (expect POSITIVE: dense = stable SHAP):")
m2 = ols("A_Stability", ["density_pct"], sub)
print(f"  coef={m2.params['density_pct']:+.3f}  p={m2.pvalues['density_pct']:.2e}  R2={m2.rsquared:.3f}")

print("M3 — density -> NonFragility (expect NEGATIVE: dense = fragile):")
m3 = ols("A_NonFrag", ["density_pct"], sub)
print(f"  coef={m3.params['density_pct']:+.3f}  p={m3.pvalues['density_pct']:.2e}  R2={m3.rsquared:.3f}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — M4: mediation — does controlling density collapse the GROUP gaps?
# For Stability and NonFragility: compare the group coefficient WITHOUT vs WITH
# density as a control. If the group effect shrinks toward 0 when density is added,
# density MEDIATES the group difference (the common-cause story holds).
# ─────────────────────────────────────────────────────────────────────────
print("M4 — mediation by density (group effect before vs after controlling density):")
for axis in ["A_Stability", "A_NonFrag"]:
    m_base = ols(axis, ["is_dis"], sub)
    m_med  = ols(axis, ["is_dis","density_pct"], sub)
    b0 = m_base.params["is_dis"]; p0 = m_base.pvalues["is_dis"]
    b1 = m_med.params["is_dis"];  p1 = m_med.pvalues["is_dis"]
    shrink = (b0 - b1) / b0 * 100 if b0 != 0 else np.nan
    print(f"\n  {axis}:")
    print(f"    group effect WITHOUT density: {b0:+.3f} (p={p0:.2e})")
    print(f"    group effect WITH    density: {b1:+.3f} (p={p1:.2e})")
    print(f"    shrinkage = {shrink:.0f}%  "
          f"-> {'density MEDIATES' if abs(b1)<abs(b0)*0.5 else 'partial/none'}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Formal mediation test (Sobel-style) for each axis
# indirect effect = a (dis->density) * b (density->axis | dis)
# ─────────────────────────────────────────────────────────────────────────
print("M5 — indirect (mediated) effect via density:")
# a-path: is_dis -> density
m_a = ols("density_pct", ["is_dis"], sub)
a, sa = m_a.params["is_dis"], m_a.bse["is_dis"]
for axis in ["A_Stability", "A_NonFrag"]:
    # b-path: density -> axis controlling is_dis
    m_b = ols(axis, ["density_pct","is_dis"], sub)
    b, sb = m_b.params["density_pct"], m_b.bse["density_pct"]
    indirect = a * b
    se = np.sqrt(b**2 * sa**2 + a**2 * sb**2)
    z = indirect / se
    p = 2*(1 - stats.norm.cdf(abs(z)))
    print(f"  {axis:14} indirect={indirect:+.4f}  z={z:+.2f}  p={p:.2e}  "
          f"{'significant mediation' if p<0.05 else 'n.s.'}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Mechanism verdict
# ─────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("R-14 MECHANISM VERDICT")
print("=" * 70)
dens_stab = m2.params['density_pct'] > 0 and m2.pvalues['density_pct'] < 0.05
dens_frag = m3.params['density_pct'] < 0 and m3.pvalues['density_pct'] < 0.05
print(f"M1 disadvantaged denser        : {'YES' if p_d<0.05 else 'no'} (p={p_d:.1e})")
print(f"M2 density -> Stability (+)     : {'YES' if dens_stab else 'no'}")
print(f"M3 density -> Fragility (NonF-) : {'YES' if dens_frag else 'no'}")
print("-" * 70)
if p_d < 0.05 and dens_stab and dens_frag:
    print("MECHANISM CONFIRMED: density is a COMMON CAUSE of both stability and")
    print("  fragility. The disadvantaged group is denser; density makes SHAP")
    print("  consistent (stable) AND places approvals in the typicality-paradox")
    print("  risky region (fragile). 'Stable yet fragile' is ONE mechanism, not a")
    print("  contradiction. R-14 resolved.")
else:
    print("MECHANISM PARTIAL: density does not fully explain both paths.")
    print("  Report what holds; seek an additional factor for the unexplained path.")
print("=" * 70)

Borderline: dis_primary=254, advantaged=224
M1 — density by group (raw density percentile; higher=denser):
  dis_primary mean density = 0.550
  advantaged  mean density = 0.517
  Mann-Whitney (dis>adv) p = 1.636e-01  -> no density difference
M2 — density -> Stability (expect POSITIVE: dense = stable SHAP):
  coef=+0.236  p=1.73e-11  R2=0.091
M3 — density -> NonFragility (expect NEGATIVE: dense = fragile):
  coef=-0.522  p=1.67e-68  R2=0.474
M4 — mediation by density (group effect before vs after controlling density):

  A_Stability:
    group effect WITHOUT density: +0.086 (p=2.38e-05)
    group effect WITH    density: +0.079 (p=5.54e-05)
    shrinkage = 9%  -> partial/none

  A_NonFrag:
    group effect WITHOUT density: -0.072 (p=2.71e-04)
    group effect WITH    density: -0.055 (p=1.28e-04)
    shrinkage = 24%  -> partial/none
M5 — indirect (mediated) effect via density:
  A_Stability    indirect=+0.0075  z=+1.23  p=2.18e-01  n.s.
  A_NonFrag      indirect=-0.0170  z=-1.25  p=2.11e-